# 1. Train the classifier (GPU)
Runtime > Change runtime type > GPU. Uses Colab's default PyTorch. Output: `waste_classifier.pth` and `classes.json` in Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!rm -rf /content/raw /content/dataset
!unzip -q -o "/content/drive/MyDrive/Waste assistant/dataset.zip" -d /content/
!mv /content/raw /content/dataset   # skip this line if the zip already unpacks to "dataset"
!ls /content/dataset

## Build the 8-class folder (`data8`) from the 4 group folders

In [ ]:
# Colab cell: turns your 4-group folders into the 8 training classes (data8), cleaned and balanced.
import os, shutil, random, hashlib, re
from PIL import Image

SRC = "dataset"     # your 4 group folders: biodegradable, recyclable, e waste, hazardous
DST = "data8"       # the folder the training notebook should read
CAP = 400           # most images kept per class (None = keep all)
random.seed(42)

def norm(s):        # "E Waste", "e-waste", "ewaste" all become "ewaste"
    return re.sub(r"[^a-z0-9]", "", s.lower())

def to_class(parts):
    """parts = folder names from the group folder down to the image."""
    top = norm(parts[0]); rest = [norm(x) for x in parts[1:-1]]
    if top.startswith("bio"):  return "organic"
    if top.startswith(("ewaste", "electr", "waste")) or top == "e": return "ewaste"
    if top.startswith("haz"):  return "hazardous"
    if top.startswith("rec"):
        if any(x in ("cardboard", "paper") for x in rest): return "paper_cardboard"
        if "glass" in rest: return "glass"
        if "metal" in rest: return "metal"
        if "plastic" in rest:
            if any(x in ("packets", "packet", "film") for x in rest): return "plastic_film"
            return "plastic_bottle_container"          # PET and HDPE
    return None

found, unmapped, seen, bad, dup = {}, [], set(), 0, 0
for root, _, files in os.walk(SRC):
    for f in files:
        if not f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")): continue
        path = os.path.join(root, f)
        parts = os.path.relpath(path, SRC).split(os.sep)
        cls = to_class(parts)
        if cls is None: unmapped.append(path); continue
        try:
            with Image.open(path) as im: im.verify()         # skip broken files
        except Exception:
            bad += 1; continue
        h = hashlib.md5(open(path, "rb").read()).hexdigest()
        if h in seen: dup += 1; continue                    # skip exact duplicates
        seen.add(h)
        sub = "/".join(parts[1:-1])                         # sub-folder, used to balance
        found.setdefault(cls, {}).setdefault(sub, []).append(path)

shutil.rmtree(DST, ignore_errors=True)
for cls, subs in found.items():
    os.makedirs(f"{DST}/{cls}")
    for l in subs.values(): random.shuffle(l)
    chosen, i = [], 0
    while any(subs.values()) and (CAP is None or len(chosen) < CAP):   # take evenly from every sub-folder
        for l in subs.values():
            if l and (CAP is None or len(chosen) < CAP): chosen.append(l.pop())
    for k, p in enumerate(chosen):
        shutil.copy(p, f"{DST}/{cls}/{k}_{os.path.basename(p)}")

print("broken skipped:", bad, " duplicates skipped:", dup, " unmapped files:", len(unmapped))
for p in unmapped[:10]: print("  unmapped:", p)
for c in sorted(os.listdir(DST)): print(f"{c:28s}", len(os.listdir(f'{DST}/{c}')))


## Train, validate, test and save
12 epochs, AdamW lr 5e-4, cosine schedule, class-weighted loss, 70/15/15 random split.

In [ ]:
import os, torch, torch.nn as nn, numpy as np, json
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix

OUT = 'data8'
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev)

mean, std = [0.485,0.456,0.406], [0.229,0.224,0.225]
tr = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.3,0.3,0.3),
    transforms.ToTensor(), transforms.Normalize(mean,std)])
ev = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(), transforms.Normalize(mean,std)])

# Load the entire dataset from the processed data8 directory
full_dataset = datasets.ImageFolder(OUT)
K = len(full_dataset.classes)
print(full_dataset.classes)

# Set random seed for reproducibility
torch.manual_seed(42)

# Split into train (70%), validation (15%), and test (15%)
n_total = len(full_dataset)
n_train = int(0.70 * n_total)
n_val = int(0.15 * n_total)
n_test = n_total - n_train - n_val

train_ds, val_ds, test_ds = torch.utils.data.random_split(full_dataset, [n_train, n_val, n_test])

# Override transforms for splits
train_ds.dataset.transform = tr
val_ds.dataset.transform = ev
test_ds.dataset.transform = ev

train_dl = torch.utils.data.DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_dl   = torch.utils.data.DataLoader(val_ds, batch_size=64, num_workers=2)
test_dl  = torch.utils.data.DataLoader(test_ds, batch_size=64, num_workers=2)

# Class weights computed from training targets
train_targets = [full_dataset.targets[i] for i in train_ds.indices]
cnt = np.bincount(train_targets, minlength=K)
w = torch.tensor(len(train_ds)/(K*cnt), dtype=torch.float).to(dev)

model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.DEFAULT)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, K)
model.to(dev)

EPOCHS = 12
opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
loss_fn = nn.CrossEntropyLoss(weight=w)

def evaluate(dl):
    model.eval(); P=[]; L=[]
    with torch.no_grad():
        for x,y in dl:
            P += model(x.to(dev)).argmax(1).cpu().tolist(); L += y.tolist()
    return np.mean(np.array(P)==np.array(L)), P, L

best, best_state = 0, None
for ep in range(EPOCHS):
    model.train()
    for x,y in train_dl:
        x,y = x.to(dev), y.to(dev)
        opt.zero_grad(); loss_fn(model(x), y).backward(); opt.step()
    sched.step()
    acc,_,_ = evaluate(val_dl)
    print(f'epoch {ep+1}/{EPOCHS}  val acc {acc:.3f}')
    if acc > best:
        best = acc
        best_state = {k: v.clone() for k,v in model.state_dict().items()}

model.load_state_dict(best_state)
acc, P, L = evaluate(test_dl)
print(f'\nTEST accuracy: {acc:.3f}\n')
print(classification_report(L, P, target_names=full_dataset.classes, digits=3))
print(confusion_matrix(L, P))

SAVE = '/content/drive/MyDrive/Waste assistant'
os.makedirs(SAVE, exist_ok=True)
torch.save(best_state, f'{SAVE}/waste_classifier.pth')
json.dump(full_dataset.classes, open(f'{SAVE}/classes.json','w'))
print('saved to Drive')